# 01 · Ingesta y unificación (R1 · Hito 2)
**Proyecto semestral · Sala de crisis COVID-19 · Gestión de Datos IN1232C · 2026-2**

1. **Descargar** los 1.143 reportes diarios, las series de tiempo y la tabla UID (capa **cruda**: CSV tal cual).
2. **Unificar** los cinco esquemas en una sola tabla con nombres en snake_case y fechas ISO.
3. Guardar la capa **limpia** en Parquet.
4. **Comparar** tamaño en disco y tiempo de lectura entre CSV y Parquet.

Los datos quedan en Google Drive, para que los notebooks 02 y 03 los lean sin descargar todo de nuevo.

## Setup

In [1]:
import os
import time
import requests
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

In [2]:
# Montar Google Drive (pide autorizar la cuenta la primera vez)
from google.colab import drive
drive.mount('/content/drive')

PROYECTO = '/content/drive/MyDrive/proyecto-gestion-datos-2026'
CRUDOS   = '/content/datos_crudos'                       # capa cruda: disco de la sesión (no se sube a GitHub)
LIMPIOS  = os.path.join(PROYECTO, 'data', 'clean')       # capa limpia: Parquet en Drive

os.makedirs(os.path.join(CRUDOS, 'daily_reports'), exist_ok=True)
os.makedirs(os.path.join(CRUDOS, 'time_series'), exist_ok=True)
os.makedirs(LIMPIOS, exist_ok=True)

## 1 · Descarga de los datos crudos
Igual que con el CSV de la NBA: `requests.get()` y guardar el contenido en un archivo. Son 1.143 archivos, así que **tarda varios minutos**; si se corta, al volver a ejecutar solo baja los que faltan.

In [3]:
URL = 'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/'

def descargar(url, ruta):
    # Descarga un archivo solo si todavía no está en disco
    if not os.path.exists(ruta):
        response = requests.get(url)
        if response.status_code != 200:
            print('ERROR al descargar', url)
            return
        with open(ruta, 'wb') as f:
            f.write(response.content)

In [4]:
# Reportes diarios: uno por día entre el 22-ene-2020 y el 9-mar-2023 (repositorio archivado)
fechas = pd.date_range('2020-01-22', '2023-03-09')
inicio = time.time()

for i, fecha in enumerate(fechas):
    nombre = fecha.strftime('%m-%d-%Y') + '.csv'
    descargar(URL + 'csse_covid_19_daily_reports/' + nombre,
              os.path.join(CRUDOS, 'daily_reports', nombre))
    if i % 100 == 0:
        print(f'{i} de {len(fechas)} archivos...')

archivos = sorted(os.listdir(os.path.join(CRUDOS, 'daily_reports')))
print(f'{len(archivos)} reportes en disco ({time.time() - inicio:.0f} s)')
assert len(archivos) == 1143, 'Faltan reportes: vuelva a ejecutar la celda'

0 de 1143 archivos...
100 de 1143 archivos...
200 de 1143 archivos...
300 de 1143 archivos...
400 de 1143 archivos...
500 de 1143 archivos...
600 de 1143 archivos...
700 de 1143 archivos...
800 de 1143 archivos...
900 de 1143 archivos...
1000 de 1143 archivos...
1100 de 1143 archivos...
1143 reportes en disco (0 s)


In [5]:
# Series de tiempo (formato ancho) y tabla UID (población, coordenadas e ISO3 de cada territorio)
series = ['time_series_covid19_confirmed_global.csv', 'time_series_covid19_deaths_global.csv',
          'time_series_covid19_recovered_global.csv', 'time_series_covid19_confirmed_US.csv',
          'time_series_covid19_deaths_US.csv']
for s in series:
    descargar(URL + 'csse_covid_19_time_series/' + s, os.path.join(CRUDOS, 'time_series', s))

descargar(URL + 'UID_ISO_FIPS_LookUp_Table.csv', os.path.join(CRUDOS, 'UID_ISO_FIPS_LookUp_Table.csv'))
print('Series y tabla UID descargadas')

Series y tabla UID descargadas


## 2 · Los cinco esquemas
Recorremos los archivos en orden y anotamos cada vez que cambia la lista de columnas.

In [6]:
fechas_archivo = [pd.to_datetime(a[:10], format='%m-%d-%Y') for a in archivos]
archivos = [a for _, a in sorted(zip(fechas_archivo, archivos))]    # ordenar por fecha, no alfabéticamente

etapas = []
columnas_anteriores = None
for a in archivos:
    columnas = list(pd.read_csv(os.path.join(CRUDOS, 'daily_reports', a), nrows=0).columns)
    if columnas != columnas_anteriores:
        etapas.append({'desde': a, 'n_columnas': len(columnas), 'columnas': ', '.join(columnas)})
        columnas_anteriores = columnas

pd.DataFrame(etapas, index=range(1, len(etapas) + 1))

,desde,n_columnas,columnas
1,01-22-2020.csv,6,"Province/State, Country/Region, Last Update, C..."
2,03-01-2020.csv,8,"Province/State, Country/Region, Last Update, C..."
3,03-22-2020.csv,12,"FIPS, Admin2, Province_State, Country_Region, ..."
4,05-29-2020.csv,14,"FIPS, Admin2, Province_State, Country_Region, ..."
5,11-09-2020.csv,14,"FIPS, Admin2, Province_State, Country_Region, ..."


## 3 · Unificación
Un solo diccionario traduce **todos** los nombres de los cinco esquemas a snake_case:

| Problema | Decisión |
|---|---|
| `Country/Region` y `Country_Region`, `Last Update` y `Last_Update`… | Mismo nombre en snake_case |
| `Latitude`/`Longitude` (etapa 2) y `Lat`/`Long_` (etapa 3+) | `lat`, `long` |
| `Incidence_Rate` (etapa 4) e `Incident_Rate` (etapa 5) | `incident_rate` |
| ¿Qué fecha tiene cada fila? | `fecha_reporte` = fecha del **nombre del archivo** |
| Linaje | `archivo_origen` y `fila_origen` (n.º de fila dentro del CSV) |

Los nombres de país **no** se corrigen aquí: eso se hace en el notebook de calidad.

In [7]:
renombrar = {
    'Province/State': 'province_state', 'Province_State': 'province_state',
    'Country/Region': 'country_region', 'Country_Region': 'country_region',
    'Last Update': 'last_update',       'Last_Update': 'last_update',
    'Latitude': 'lat', 'Lat': 'lat',    'Longitude': 'long', 'Long_': 'long',
    'Confirmed': 'confirmed', 'Deaths': 'deaths', 'Recovered': 'recovered', 'Active': 'active',
    'FIPS': 'fips', 'Admin2': 'admin2', 'Combined_Key': 'combined_key',
    'Incidence_Rate': 'incident_rate',  'Incident_Rate': 'incident_rate',
    'Case-Fatality_Ratio': 'case_fatality_ratio', 'Case_Fatality_Ratio': 'case_fatality_ratio',
}

columnas_finales = ['fecha_reporte', 'fips', 'admin2', 'province_state', 'country_region', 'combined_key',
                    'lat', 'long', 'confirmed', 'deaths', 'recovered', 'active', 'incident_rate',
                    'case_fatality_ratio', 'last_update', 'archivo_origen', 'fila_origen']

def leer_reporte(nombre):
    df = pd.read_csv(os.path.join(CRUDOS, 'daily_reports', nombre))
    df = df.rename(columns=renombrar)
    df['fecha_reporte'] = pd.to_datetime(nombre[:10], format='%m-%d-%Y')
    df['archivo_origen'] = nombre
    df['fila_origen'] = range(1, len(df) + 1)
    return df.reindex(columns=columnas_finales)      # columnas que no existían ese día quedan como NaN

inicio = time.time()
lista = []
for i, a in enumerate(archivos):
    lista.append(leer_reporte(a))
    if i % 200 == 0:
        print(f'{i} de {len(archivos)} archivos leídos...')

covid = pd.concat(lista, ignore_index=True)
del lista                                          # liberar memoria
print(f'Tabla unificada: {covid.shape[0]:,} filas y {covid.shape[1]} columnas ({time.time() - inicio:.0f} s)')

0 de 1143 archivos leídos...


200 de 1143 archivos leídos...


400 de 1143 archivos leídos...


600 de 1143 archivos leídos...


800 de 1143 archivos leídos...


1000 de 1143 archivos leídos...


Tabla unificada: 4,287,473 filas y 17 columnas (9 s)


In [8]:
# Tipos de datos
covid['last_update'] = pd.to_datetime(covid['last_update'], format='mixed')   # varios formatos de fecha
for col in ['admin2', 'province_state', 'country_region', 'combined_key']:
    covid[col] = covid[col].str.strip()                                       # ' Azerbaijan' -> 'Azerbaijan'

covid.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4287473 entries, 0 to 4287472
Data columns (total 17 columns):
 #   Column               Dtype         
---  ------               -----         
 0   fecha_reporte        datetime64[ns]
 1   fips                 float64       
 2   admin2               object        
 3   province_state       object        
 4   country_region       object        
 5   combined_key         object        
 6   lat                  float64       
 7   long                 float64       
 8   confirmed            float64       
 9   deaths               float64       
 10  recovered            float64       
 11  active               float64       
 12  incident_rate        float64       
 13  case_fatality_ratio  float64       
 14  last_update          datetime64[ns]
 15  archivo_origen       object        
 16  fila_origen          int64         
dtypes: datetime64[ns](2), float64(9), int64(1), object(5)
memory usage: 556.1+ MB


### Verificaciones

In [9]:
# 1. No se perdió ni se duplicó ninguna fila
filas_csv = 0
for a in archivos:
    with open(os.path.join(CRUDOS, 'daily_reports', a), encoding='utf-8-sig') as f:
        filas_csv += sum(1 for _ in f) - 1                 # menos la fila de encabezado

print(f'Filas en los CSV: {filas_csv:,} | Filas en la tabla: {len(covid):,}')
assert filas_csv == len(covid), 'Se perdieron o duplicaron filas'

# 2. Están los 1.143 días
assert covid['fecha_reporte'].nunique() == 1143

# 3. Ninguna fecha de actualización quedó sin interpretar
assert covid['last_update'].notna().all()
print('Verificaciones OK')

Filas en los CSV: 4,287,473 | Filas en la tabla: 4,287,473
Verificaciones OK


In [10]:
# ¿Desde qué fecha existe cada columna? (primer día con algún valor no nulo)
desde = {}
for col in columnas_finales:
    desde[col] = covid.loc[covid[col].notna(), 'fecha_reporte'].min().date()
pd.Series(desde, name='existe desde')

fecha_reporte          2020-01-22
fips                   2020-03-22
admin2                 2020-03-22
province_state         2020-01-22
country_region         2020-01-22
combined_key           2020-03-22
lat                    2020-03-01
long                   2020-03-01
confirmed              2020-01-22
deaths                 2020-01-22
recovered              2020-01-22
active                 2020-03-22
incident_rate          2020-05-29
case_fatality_ratio    2020-05-29
last_update            2020-01-22
archivo_origen         2020-01-22
fila_origen            2020-01-22
Name: existe desde, dtype: object

In [11]:
# Linaje: de qué archivo y fila sale el pico de Chile del 17-jun-2020
covid.loc[(covid['country_region'] == 'Chile') & (covid['fecha_reporte'] == '2020-06-17'),
          ['fecha_reporte', 'province_state', 'confirmed', 'deaths', 'archivo_origen', 'fila_origen']].head()

,fecha_reporte,province_state,confirmed,deaths,archivo_origen,fila_origen
300777,2020-06-17,Antofagasta,5241.0,81.0,06-17-2020.csv,3093
300782,2020-06-17,Araucania,2815.0,36.0,06-17-2020.csv,3098
300784,2020-06-17,Arica y Parinacota,1218.0,11.0,06-17-2020.csv,3100
300791,2020-06-17,Atacama,561.0,0.0,06-17-2020.csv,3107
300795,2020-06-17,Aysen,26.0,0.0,06-17-2020.csv,3111


## 4 · Series de tiempo y tabla UID
Las series vienen en formato **ancho** (una columna por día). Con `melt` las pasamos a formato **largo** (una fila por territorio y día), que es como está la tabla unificada.

In [12]:
ts = pd.read_csv(os.path.join(CRUDOS, 'time_series', 'time_series_covid19_confirmed_global.csv'))
print('Formato ancho:', ts.shape)

ts_largo = ts.melt(id_vars=['Province/State', 'Country/Region', 'Lat', 'Long'],
                   var_name='fecha', value_name='confirmados')
ts_largo = ts_largo.rename(columns={'Province/State': 'province_state', 'Country/Region': 'country_region',
                                    'Lat': 'lat', 'Long': 'long'})
ts_largo['fecha'] = pd.to_datetime(ts_largo['fecha'], format='%m/%d/%y')
print('Formato largo:', ts_largo.shape)
ts_largo.head()

Formato ancho: (289, 1147)
Formato largo: (330327, 6)


,province_state,country_region,lat,long,fecha,confirmados
0,NaN,Afghanistan,33.94,67.71,2020-01-22,0
1,NaN,Albania,41.15,20.17,2020-01-22,0
2,NaN,Algeria,28.03,1.66,2020-01-22,0
3,NaN,Andorra,42.51,1.52,2020-01-22,0
4,NaN,Angola,-11.20,17.87,2020-01-22,0


In [13]:
uid = pd.read_csv(os.path.join(CRUDOS, 'UID_ISO_FIPS_LookUp_Table.csv'))
uid = uid.rename(columns={'UID': 'uid', 'Admin2': 'admin2', 'Province_State': 'province_state',
                          'Country_Region': 'country_region', 'Lat': 'lat', 'Long_': 'long',
                          'Combined_Key': 'combined_key', 'Population': 'population', 'FIPS': 'fips'})
print(uid.shape)
uid[uid['country_region'] == 'Chile'].head()

(4321, 12)


,uid,iso2,iso3,code3,fips,admin2,province_state,country_region,lat,long,combined_key,population
74,152,CL,CHL,152.0,NaN,NaN,NaN,Chile,-35.68,-71.54,Chile,1.91e+07
75,15201,CL,CHL,152.0,NaN,NaN,Antofagasta,Chile,-23.65,-70.40,"Antofagasta, Chile",6.08e+05
76,15202,CL,CHL,152.0,NaN,NaN,Araucania,Chile,-38.95,-72.33,"Araucania, Chile",9.57e+05
77,15203,CL,CHL,152.0,NaN,NaN,Arica y Parinacota,Chile,-18.59,-69.48,"Arica y Parinacota, Chile",2.26e+05
78,15204,CL,CHL,152.0,NaN,NaN,Atacama,Chile,-27.57,-70.05,"Atacama, Chile",2.89e+05


## 5 · Capa limpia en Parquet

In [14]:
# compression='gzip' comprime más que la opción por defecto (queda bajo los 100 MB que acepta GitHub)
covid.to_parquet(os.path.join(LIMPIOS, 'covid_diario.parquet'), index=False, compression='gzip')
ts_largo.to_parquet(os.path.join(LIMPIOS, 'serie_confirmados_global.parquet'), index=False)
uid.to_parquet(os.path.join(LIMPIOS, 'uid.parquet'), index=False)

for a in sorted(os.listdir(LIMPIOS)):
    print(f'{a:35s} {os.path.getsize(os.path.join(LIMPIOS, a)) / 1e6:6.1f} MB')

covid_diario.parquet                  70.0 MB
serie_confirmados_global.parquet       1.4 MB
uid.parquet                            0.2 MB


## 6 · CSV vs Parquet
Comparamos **los mismos datos** guardados de tres formas: los 1.143 CSV originales, un solo CSV con la tabla unificada y el Parquet.

In [15]:
# Tamaño en disco
mb_crudos  = sum(os.path.getsize(os.path.join(CRUDOS, 'daily_reports', a)) for a in archivos) / 1e6
covid.to_csv('/content/covid_diario.csv', index=False)
mb_csv     = os.path.getsize('/content/covid_diario.csv') / 1e6
mb_parquet = os.path.getsize(os.path.join(LIMPIOS, 'covid_diario.parquet')) / 1e6

# Tiempo de lectura
inicio = time.time(); x = pd.read_csv('/content/covid_diario.csv', low_memory=False); t_csv = time.time() - inicio
inicio = time.time(); x = pd.read_parquet(os.path.join(LIMPIOS, 'covid_diario.parquet')); t_parquet = time.time() - inicio
inicio = time.time(); x = pd.read_parquet(os.path.join(LIMPIOS, 'covid_diario.parquet'),
                                          columns=['fecha_reporte', 'country_region', 'confirmed']); t_cols = time.time() - inicio
del x

comparacion = pd.DataFrame({
    'tamaño_MB': [mb_crudos, mb_csv, mb_parquet, np.nan],
    'lectura_s': [np.nan, t_csv, t_parquet, t_cols],
}, index=['1.143 CSV originales', 'CSV unificado', 'Parquet', 'Parquet (solo 3 columnas)'])
comparacion.round(2)

,tamaño_MB,lectura_s
1.143 CSV originales,587.06,NaN
CSV unificado,747.61,8.29
Parquet,70.01,1.11
Parquet (solo 3 columnas),NaN,0.22


In [16]:
print(f'Parquet ocupa {mb_csv / mb_parquet:.0f} veces menos que el CSV unificado')
print(f'Parquet se lee {t_csv / t_parquet:.0f} veces más rápido que el CSV unificado')

Parquet ocupa 11 veces menos que el CSV unificado
Parquet se lee 7 veces más rápido que el CSV unificado


**Interpretación:** con los mismos datos, Parquet ocupa unas **10 veces menos** que el CSV unificado (70 MB frente a 748 MB) y se lee **entre 3 y 7 veces más rápido** (y bastante más si se leen solo 3 columnas; las cifras exactas cambian según la máquina). Además conserva los tipos (fechas, números), así que no hay que volver a convertirlos. Y como guarda los datos **por columna**, puede leer solo las columnas que se necesitan; un CSV siempre se lee completo. Por eso la capa limpia es Parquet.

## Resumen
* **Capa cruda:** 1.143 reportes + series + tabla UID descargados con código (no se suben al repositorio).
* **Capa limpia:** `covid_diario.parquet` con los 5 esquemas unificados, fechas ISO y linaje por fila (archivo y fila de origen).
* Siguiente: `02_calidad` (reglas de las 6 dimensiones) y `03_dw_estrella` (Data Warehouse).